# Session 8 · Predicting age from sleep EEG (machine learning)

Sleep changes with age: less N3, more awakenings, fewer spindles, a flatter spectrum. So a model can **predict a person's age from their night**. The gap between predicted and true age is called the *brain age gap*, and it is studied as a marker of brain health.

You get **all the features** of the week in one table (`precomputed/cohort_features.csv`, one row per night, about 230 features). We use one model, a **Random Forest**, and start with a **weak starter set** of features (eight basic sleep statistics) that you will **improve**. The point is not the best model, it is to **evaluate honestly**, and to see that choosing features is also a form of fitting. You will:
1. set a **baseline** (what if we always predict the mean age?)
2. fit a first model, score it with **cross-validation by person**, and **check that no person is in two folds**
3. see what happens when you split **by night** instead (leakage)
4. **tune and score in one go** with *nested* cross-validation
5. **improve the feature set**, and check it on 16 people that were locked away
6. (extra) test whether a model trained on night 1 works on **night 2**

**Course rules for this notebook**
- The unit is the **person**. 75 of the 78 people have two nights, and the two nights of one person look alike. Both nights of a person must always be on the **same side** of a split.
- Anything that learns from data (filling missing values, choosing hyperparameters, selecting features) is **fitted on the training part only**. We put it all in a `Pipeline` so that this cannot go wrong.
- Metric: **MAE** in years (easy to read) and R².

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 0

## 1 · The table, and a starter set of features

One row per night. `age` is what we predict (the **target**); `subject` tells us who the night belongs to.

**Locked test people.** Before we look at anything, we set aside **16 people (20 %)**. They are in the table `df_locked` and **nobody touches them until Exercise 5.5**. Everything else (62 people, `df`) is for developing and comparing. Why: in Section 5 you will try many feature sets and keep the best. Each try is a peek at the scores, so the best score after many tries is too optimistic. The locked people give one score that was not used to choose anything.

**Starter features.** The table has about 230 features. We start with only **eight basic sleep statistics**: total sleep time, sleep efficiency, sleep onset latency, WASO and the percentage of each stage. That is what a clinical sleep report gives. It is not optimal, and improving it is your job in Section 5.

**Missing values** are real: a person with no N3 has no N3 band power. We don't drop those nights (we have few), we fill the gaps **inside the pipeline**. (Why they are missing: Exercise 5.6.)

In [ ]:
df_all = pd.read_csv("../precomputed/cohort_features.csv")

# Lock away 20 % of the PEOPLE (both nights of each), once, with a fixed seed
dev_idx, locked_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
                           .split(df_all, groups=df_all["subject"]))
df = df_all.iloc[dev_idx].reset_index(drop=True)
df_locked = df_all.iloc[locked_idx].reset_index(drop=True)
print(f"development: {len(df)} nights, {df['subject'].nunique()} people | locked: {len(df_locked)} nights, {df_locked['subject'].nunique()} people")

# All usable features: everything numeric except identifiers and the target. We leave out the columns that compare
# YASA with the expert (yasa_accuracy, yasa_kappa, yasa_F1_*): they need an expert hypnogram, and they are not a property of the night.
drop = ["code", "subject", "night", "age", "sex", "ART", "UNS"]
drop += [c for c in df.columns if c.startswith(("yasa_accuracy", "yasa_kappa", "yasa_F1"))]
ALL_FEATURES = df.drop(columns=drop).select_dtypes("number").columns.tolist()

STARTER = ["TST", "SE", "SOL", "WASO", "%N1", "%N2", "%N3", "%REM"]

X = df[STARTER]
y = df["age"].to_numpy()
groups = df["subject"].to_numpy()  # who each night belongs to
print(len(ALL_FEATURES), "features available,", len(STARTER), "in the starter set")

## 2 · A baseline, and the first honest score

Any model needs a **baseline**, or "MAE = 14 years" means nothing. The simplest one always predicts the training mean.

We score with **cross-validation by person**: `GroupKFold` makes 5 folds so that all nights of a person fall in the same fold. In each round, the model is fitted on 4 folds and predicts the 5th.

In [ ]:
def scores(y_true, y_pred):
    return {"MAE": round(mean_absolute_error(y_true, y_pred), 2), "R2": round(r2_score(y_true, y_pred), 2)}


outer = GroupKFold(n_splits=5)

baseline = DummyRegressor(strategy="mean")
pred_baseline = cross_val_predict(baseline, X, y, groups=groups, cv=outer)
print("baseline (mean age):", scores(y, pred_baseline))

**Exercise 2.1 — a first model.** Build a **pipeline** with two steps: fill missing values with the median (`SimpleImputer(strategy="median")`) and a `RandomForestRegressor(n_estimators=200, max_features=0.3, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)`. Score it with the same `outer` split.

Why a pipeline? `cross_val_predict` refits the *whole* pipeline on each training part. The median used to fill gaps is then computed without the test fold. (A forest doesn't need scaled features. With a model that does, a `StandardScaler` would go into the pipeline too, for the same reason.)

In [ ]:
# TODO: pipeline = imputer -> RandomForestRegressor(...)
forest = ...
pred_forest = cross_val_predict(forest, X, y, groups=groups, cv=outer)
print("random forest, starter set:", scores(y, pred_forest))

**Exercise 2.2 — no person in two folds.** `GroupKFold` is supposed to keep each person on one side, but **never trust, check**. Write a function `overlap(cv, X, y, groups)` that returns, for each fold of `cv`, the **number of people that appear in both the training and the test part**. Run it on `outer`: you need `[0, 0, 0, 0, 0]`.

Then, as an assert that stops the notebook if it ever fails, `assert sum(...) == 0`. Put such a check in every notebook where you split people.

In [ ]:
def overlap(cv, X, y, groups):
    """For each fold: how many people are in both the training and the test part?"""
    # TODO: for each (train, test) pair of cv.split(X, y, groups), count the people (groups[...]) in both
    return ...


print("overlap per fold:", overlap(outer, X, y, groups))
# TODO: an assert that fails if any fold has an overlap

## 3 · What a wrong split does

Now do it the wrong way on purpose: a plain `KFold` with shuffling splits the **nights** at random, so the two nights of the same person often end up on both sides (one in training, one in test).

**Exercise 3.1.** Score the same forest with `KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` and compare it with the split by person. Do it for the **starter set** and for **all features**: how much a leak helps depends on how many features the model has to memorise people with. Then use your `overlap` function on the leaky split.

In [ ]:
leaky = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# TODO: score the forest with the `outer` split (by person) and the `leaky` split (by night),
# for X (starter) and for df[ALL_FEATURES]
rows = {}
print(pd.DataFrame(rows).T)

# TODO: use `overlap` on the leaky split

## 4 · Tuning: nested cross-validation

The forest settings above were a guess. To choose them we need to try several and keep the best. But if we pick the best on **the same folds we report**, the score is optimistic: we selected the winner by looking at the test folds.

**Nested cross-validation** keeps the two jobs apart:
- the **inner loop** (on the training part only) chooses the hyperparameters: `GridSearchCV` with its own `GroupKFold`
- the **outer loop** scores the chosen model on people it has never seen

```
outer fold 1: train = people A  → inner CV on A picks the settings → refit on A → score on test people (never used)
outer fold 2: ...
```
The reported number is the average over the outer test folds. The settings **chosen** in each outer fold can differ: that's fine, we are evaluating the *procedure* "tune, then fit".

In [ ]:
def nested_cv(make_search, X, y, groups, n_outer=5):
    """Nested CV by person. `make_search()` returns a fresh GridSearchCV. Returns predictions and the chosen settings."""
    pred = np.zeros(len(y))
    chosen = []
    for train, test in GroupKFold(n_splits=n_outer).split(X, y, groups):
        search = make_search()
        search.fit(X.iloc[train], y[train], groups=groups[train])  # inner folds are also by person
        pred[test] = search.predict(X.iloc[test])
        chosen.append(search.best_params_)
    return pred, chosen

**Exercise 4.1 — tune the forest.** Complete the search: imputer + `RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)`; try `max_features` in `[0.3, 0.6]` (the share of features each split may look at) and `min_samples_leaf` in `[1, 3]` (the smallest leaf; larger leaves = a smoother model); an inner `GroupKFold(4)` and MAE as the score (`scoring="neg_mean_absolute_error"`).

The name of a pipeline parameter is `<step name>__<parameter>`: for the forest in a `make_pipeline` it is `randomforestregressor__max_features`.

In [ ]:
def make_rf_search():
    pipe = ...  # TODO: imputer -> RandomForestRegressor(...)
    grid = ...  # TODO: max_features [0.3, 0.6], min_samples_leaf [1, 3]
    return ...  # TODO: GridSearchCV with an inner GroupKFold(4)


pred_nested, chosen = nested_cv(make_rf_search, X, y, groups)
print("random forest, nested CV, starter set:", scores(y, pred_nested))
for settings in chosen:
    print(settings)

**Exercise 4.2 — the inner folds too.** The inner `GroupKFold(4)` also splits people. Use `overlap` to check that the inner folds have no person in both parts, for each of the 5 outer training parts (`GroupKFold(5).split(X, y, groups)`: apply `overlap` to `X.iloc[train]`, `y[train]`, `groups[train]` with a `GroupKFold(4)`). Print the result.

In [ ]:
# TODO: for each outer training part, check the inner GroupKFold(4) with `overlap`

## 5 · Improve the features

The starter set is weak on purpose. Your task: **find a better set of features**, using what you know about how sleep changes with age (less deep sleep and slow waves, fewer spindles, a flatter spectrum, more fragmented sleep, ...).

**Rules**
1. Every try is scored with the **same nested CV** (`evaluate` below, on the 62 development people).
2. Choose features for a **reason** you can write down. Don't just add everything that raises the score.
3. Count your tries. After 20 tries the best score is partly luck, and only the locked people (5.5) tell you how much.

In [ ]:
def evaluate(features, make_search=make_rf_search):
    """Nested CV by person on the development people, for a list of feature names."""
    pred, _ = nested_cv(make_search, df[features], y, groups)
    return scores(y, pred)


def family(name):
    if name.startswith("yasa_"):
        return "YASA staging"
    if name.startswith(("sp_", "sw_")):
        return "spindles / slow waves"
    if name.split("_")[0] in ("delta", "theta", "alpha", "sigma", "beta", "gamma"):
        return "band power"
    return "sleep statistics"


fam = pd.Series({f: family(f) for f in ALL_FEATURES})
print(fam.value_counts())
print("starter:", evaluate(STARTER))

**Question 5.1.** The table has about 230 features for 62 people. Why is "use them all" not automatically the answer?

**Exercise 5.2 — one family at a time.** Add **one family** of features to the starter set (`STARTER + list(fam.index[fam == name])`) and score it, for each of the four families. Also score each family **alone** and **all features together**. Which family helps most? Does everything together beat the best single addition?

In [ ]:
# TODO: for each family: score "starter + family" and "family alone"; also score ALL_FEATURES
rows = {}
pd.DataFrame(rows).T

**Exercise 5.3 — your own feature set.** Build a list `MY_FEATURES` from the starter set plus the features you think matter, with a reason for each (write it as a comment). You can pick whole families, one stage (e.g. only N3), one channel (only `Fpz`), or single columns. Score it with `evaluate`.

Hints: the column names follow `<band>_<abs|rel>_<stage>_<channel>` (e.g. `delta_rel_N3_Fpz`) and `sp_`/`sw_` for spindles and slow waves. Absolute power depends on the skull and the electrode contact; relative power is less sensitive to that.

In [ ]:
# TODO: choose your features (and write a reason for each group as a comment)
MY_FEATURES = STARTER + []
print(len(MY_FEATURES), "features:", evaluate(MY_FEATURES))

**Exercise 5.4 — let the pipeline choose.** Choosing by hand is one option; the other is to let the model choose, **inside the pipeline**, so that the choice is made on the training part only. `SelectKBest(f_regression, k=...)` keeps the `k` features most correlated with age; `k` is then tuned in the inner loop together with the forest's `min_samples_leaf`.

**Careful:** picking the 30 best features on the *whole* table and then running cross-validation is a classic leak: the selection has already seen the test people.

Complete the search over `k` in `[10, 30, 60]` and `min_samples_leaf` in `[1, 3]`, on all features.

In [ ]:
def make_kbest_search():
    pipe = ...  # TODO: imputer -> SelectKBest(f_regression) -> RandomForestRegressor(n_estimators=200, max_features=0.3, ...)
    grid = ...  # TODO: selectkbest__k [10, 30, 60], randomforestregressor__min_samples_leaf [1, 3]
    return ...


print("all features + SelectKBest:", evaluate(ALL_FEATURES, make_kbest_search))

**Exercise 5.5 — the locked people (once).** Now fit the **whole procedure** (tuning included) on all 62 development people and score the 16 locked people, for the starter set, your set and the automatic selection. Do this **once**. If you go back and change `MY_FEATURES` after seeing this, the locked people are no longer locked.

In [ ]:
# TODO: fit the search on the development people and score df_locked, for the three sets
# (starter, MY_FEATURES, ALL_FEATURES with make_kbest_search)

**Question 5.5.** Your development score and the locked score differ. Which one do you report, and what does it tell you if your *hand-picked* set does clearly worse on the locked people than on the development people?

**Exercise 5.6 — why are features missing?** Some nights have missing values (`NaN`). They are not random failures. Find out the reason for three groups of columns:
1. `delta_abs_N3_Fpz` (all N3 band power): list the nights where it is missing, with `age`, `N3` (minutes of N3 by the expert) and `clean_min_N3`.
2. `sw_ptp_Fpz` (average slow-wave size): list the nights where it is missing, with `age`, `clean_min_N2`, `sw_density_N2_Fpz`.
3. `sw_ptp_Pz`: how many nights are missing, and what is `sw_density_N2_Pz` there? Is missingness related to age?

Use `df_all` (all 153 nights).

In [ ]:
# TODO: look at the three groups of missing values (use df_all)

## 6 · Extra: does it work on the second night?

The cohort has two nights per person (night 2 is the person's second recording; a first night in a new setting is known to differ from later ones, so night 1 and night 2 are not interchangeable). Different ways to split the data answer **different questions**:

| split | question it answers |
|---|---|
| (a) train on night 1, test on night 2 of the **same people** | can we recognise this person's age from a second recording? (the model has seen the person) |
| (b) train on night 1 of some people, test on night 2 of **other people** | does the model work on new people *and* a new night? |
| (c) train on night 2, test on night 1 of **other people** | the same, in the other direction (first-night effect?) |

**Exercise 6.1.** Use a forest (`forest`, settings of Exercise 2.1) with your feature set `MY_FEATURES`, on the development people. For (b) and (c), draw 5 random splits of **people** into 80 % train and 20 % test (`GroupShuffleSplit`) and average the MAE: one split of 13 people is too noisy to compare. Use `overlap` to check each split.

In [ ]:
night = df["night"].to_numpy()
X = df[MY_FEATURES]


def fit_score(train_mask, test_mask, model=forest):
    model.fit(X[train_mask], y[train_mask])
    return mean_absolute_error(y[test_mask], model.predict(X[test_mask]))


# TODO (a): train on all night-1 nights, test on all night-2 nights (same people), and the other way round

# TODO (b), (c): for 5 GroupShuffleSplit splits of people (test_size=0.2), train on `train_night` of the training
# people, test on `test_night` of the test people. Compare night 1 -> 2, night 2 -> 1 and the same-night controls.
# Check each splitter with `overlap`.

## Wrap-up

- The headline number is the **nested, subject-wise** MAE, compared with the **mean-age baseline**.
- **Check** that no person is in two folds (`overlap`), and keep the check in the notebook.
- A random split of nights (or epochs!) gives a better number and a worse model, and the leak grows with the number of features.
- Hyperparameters are chosen **inside** the training part; imputing and feature selection are in the **pipeline**.
- **Choosing features is fitting too.** Every set you try and compare is a peek at the scores: keep people locked away for one final check, or put the selection inside the pipeline.
- Features beat settings: tuning the forest changed little, the feature set changed the error from 14.5 to about 11 years.
- Missing values are not random: here they mean "no N3" or "no slow waves", which is a sign of age.

**Extra ideas:** `add_indicator=True` in the imputer; average the two nights of a person before the final score (one prediction per person); `RepeatedGroupKFold` to see the spread of the score; `permutation_importance` on the held-out folds; another model (Ridge, LightGBM) with the same pipeline and the same folds; add `sex` as a feature.

Deep learning on the raw signal: see the lecture. It needs the same rules (split by person, tune inside the training part, a baseline), and much more data than 78 people.